# ProtRL CSV workflow: SFT + GRPO on Protein Sequences (Experimentalist Workflow)

This notebook provides a clean, essential pipeline for experimentalists to fine-tune protein language models (like ProtGPT3) using reinforcement learning based on custom experimental results (CSV format).

It performs three steps:
1. **Supervised Fine-Tuning (SFT) Warm-Up**: Trains on sequences with rewards strictly higher than the dataset's mean reward. Under the hood, this uses TRL's `SFTTrainer` to automatically tokenize raw text datasets.
2. **LoRA Adapters**: Sets up parameter-efficient training to minimize memory usage.
3. **ProtRL GRPO**: Applies reinforcement learning on the complete dataset to steer the model towards higher rewards. The GRPO trainer automatically tokenizes raw sequence inputs without manual pre-tokenization cells.
4. **Sequence Generation**: Generates 1000 optimized sequences in safe batches to avoid CUDA Out-Of-Memory errors, and outputs them into a CSV.

## 1. Install and clone dependencies

In [ ]:
import subprocess
import sys
import os

# Install dependencies using standard Python to avoid linter issues
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"])
subprocess.run([sys.executable, "-m", "pip", "install", "-U", "transformers", "datasets", "peft", "accelerate", "trl", "torch", "matplotlib", "--quiet"])

# Clone repository if it doesn't already exist
if not os.path.exists("ProtRL"):
    subprocess.run(["git", "clone", "https://github.com/AI4PDLab/ProtRL.git", "--branch", "restructuring"])

## 2. Imports and basic configuration

In [ ]:
import os
import sys
import random
import csv
import torch
import pandas as pd
import matplotlib.pyplot as plt
from datasets import Dataset

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
)
import transformers
transformers.logging.set_verbosity_info()

from peft import LoraConfig, get_peft_model

# Add repository directory to path
sys.path.insert(0, "/content/ProtRL")
from src.ProtRL_Trainer import ProtRLTrainingArgument
from src.pLM_GRPO import ProtRL_GRPOTrainer

SEED = 42
MODEL_NAME = "AI4PD/ProtGPT3-112M"

random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

## 3. Load Model and Tokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)

# Ensure pad_token is set (crucial for GPT-2 models)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

n_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {n_params:,}")
print("BOS:", tokenizer.bos_token, "EOS:", tokenizer.eos_token, "PAD:", tokenizer.pad_token)

## 4. Load Custom CSV & Filter SFT Dataset with Train/Eval Split

The CSV must contain `sequence` and `reward` columns. ProtGPT3 does not require or expect sequences starting with "M" (Methionine) – they can start with any amino acid.

In [ ]:
csv_path = input("Enter the path to your CSV file: ")
df = pd.read_csv(csv_path)
print(f"Loaded {len(df)} examples.")

# Compute mean reward
mean_reward = df["reward"].mean()
print(f"Mean reward: {mean_reward:.4f}")

# Filter sequences with reward > mean
positive_df = df[df["reward"] > mean_reward].reset_index(drop=True)
print(f"Number of warm-up sequences: {len(positive_df)}")

# ProtGPT3-112M is CHARACTER-LEVEL: one token per residue, no space token.
# Pass the raw residue string - the trainer/SFT tokenizes it internally. Space-separating
# would map every space to [UNK], so training would not match what the model generates.
def format_sequence(seq):
    return str(seq).replace(" ", "")

positive_df["text"] = positive_df["sequence"].apply(format_sequence)
sft_dataset = Dataset.from_pandas(positive_df[["text"]], preserve_index=False)

# Train/Eval split for SFT dataset
sft_split = sft_dataset.train_test_split(test_size=0.2, seed=SEED, shuffle=True)

## 5. Add LoRA Adapters

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 6. Run the SFT Warm-Up Step (with Evaluation & Logging)

Under the hood, we use `SFTTrainer` from TRL to automatically format and tokenize our raw string dataset.

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

# Tokenize the raw residue strings. ProtGPT3-112M is character-level, so this produces one
# token per residue - exactly the tokens the model generates. The causal-LM collator pads
# each batch dynamically and builds the labels (labels = input_ids, pad positions -> -100).
def tokenize_sft(batch):
    return tokenizer(batch["text"], truncation=True, max_length=100)

tokenized_train = sft_split["train"].map(tokenize_sft, batched=True, remove_columns=["text"])
tokenized_eval  = sft_split["test"].map(tokenize_sft, batched=True, remove_columns=["text"])

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

sft_args = TrainingArguments(
    output_dir="./sft_results",
    report_to="none",
    logging_steps=5,
    num_train_epochs=1,
    per_device_train_batch_size=4,
    eval_strategy="steps",
    eval_steps=10,
    save_strategy="no",
    remove_unused_columns=False,
)

sft_trainer = Trainer(
    model=model,
    args=sft_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=data_collator,
    processing_class=tokenizer,  # current Trainer arg (replaces the removed `tokenizer=`)
)

sft_trainer.train()

model.save_pretrained("./lora_positive_reward_model")
tokenizer.save_pretrained("./lora_positive_reward_model")

## 7. Plot SFT Training Curves

In [ ]:
def plot_trainer_history(trainer_obj, title="SFT Warm-Up Curves"):
    history = trainer_obj.state.log_history
    train_epochs = [x["epoch"] for x in history if "loss" in x]
    train_losses = [x["loss"] for x in history if "loss" in x]
    
    plt.figure(figsize=(6, 4))
    plt.plot(train_epochs, train_losses, label="Train Loss", color="royalblue", marker="o")
    
    eval_epochs = [x["epoch"] for x in history if "eval_loss" in x]
    eval_losses = [x["eval_loss"] for x in history if "eval_loss" in x]
    if eval_losses:
        plt.plot(eval_epochs, eval_losses, label="Eval Loss", color="darkorange", marker="x")
        
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(title)
    plt.legend()
    plt.grid(True)
    plt.show()

plot_trainer_history(sft_trainer)

## 8. Generate Sequences from the SFT Model

In [ ]:
def generate_sequences(n=5):
    inputs = tokenizer("M", return_tensors="pt").to(device)

    with torch.no_grad():
        gen_ids = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=True,
            num_return_sequences=n,
            top_p=0.95,
            temperature=1.0,
            pad_token_id=tokenizer.pad_token_id,
        )

    return tokenizer.batch_decode(gen_ids, skip_special_tokens=True)

generated = generate_sequences(n=5)
for i, seq in enumerate(generated):
    print(f">{i}")
    print(seq.replace(" ", ""))
    print()

## 9. Prepare the GRPO Dataset (with Train/Eval Split)

Build the RL dataset using raw sequence and reward inputs. The trainer will automatically tokenize the completions.

### **Note on prompt grouping**
The underlying `PreferenceBatchSampler` groups training examples based on the value in the `prompt` column:
- If you use an **empty string** (`""`), all examples are treated as unconditional and are grouped together.
- If you use a **prefix or starting amino acid** (like `"M"`), they are grouped under that prefix.
- If you use a **label** (e.g. target proteins, function templates, EC numbers), you can group comparison sets by these labels, meaning the policy will only compare sequences that share the exact same target condition during learning.

In [ ]:
prompts = []
completions = []
rewards = []

for _, row in df.iterrows():
    # Here, we use "" (empty) for unconditional grouping, but you can also use functional tags/labels 
    # to group comparison sets as desired by the PreferenceBatchSampler
    prompts.append("")
    completions.append(format_sequence(row["sequence"]))
    rewards.append(float(row["reward"]))

rl_df = pd.DataFrame({
    "prompt": prompts,
    "completion": completions,
    "reward": rewards
})

rl_dataset = Dataset.from_pandas(
    rl_df[["prompt", "completion", "reward"]],
    preserve_index=False,
)

split = rl_dataset.train_test_split(test_size=0.2, seed=SEED, shuffle=True)
train_dataset = split["train"]
eval_dataset = split["test"]

print("Train examples:", len(train_dataset))
print("Eval examples:", len(eval_dataset))
print(train_dataset[0])

## 10. Run ProtRL GRPO (with Evaluation & Logging)

No manual pre-tokenization is needed; `ProtRL_GRPOTrainer` tokenizes the strings automatically.

In [ ]:
grpo_args = ProtRLTrainingArgument(
    output_dir="RL",
    report_to="none",
    logging_steps=5,
    num_train_epochs=1,
    dataloader_num_workers=1,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    eval_strategy="steps",
    eval_steps=10,
)

grpo_trainer = ProtRL_GRPOTrainer(
    model=model,
    args=grpo_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
)

grpo_trainer.train()
grpo_trainer.save_model("RL/final_model")

## 11. Plot GRPO Training Curves

In [ ]:
def plot_grpo_history(trainer_obj):
    history = trainer_obj.state.log_history
    train_epochs = [x["epoch"] for x in history if "loss" in x]
    train_losses = [x["loss"] for x in history if "loss" in x]
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    
    # Loss Plot
    ax1.plot(train_epochs, train_losses, label="Train Loss", color="crimson", marker="o")
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Loss")
    ax1.set_title("GRPO RL Loss")
    ax1.grid(True)
    ax1.legend()
    
    # Spearman Correlation metrics
    eval_epochs = [x["epoch"] for x in history if "eval_i_reward_correlation" in x]
    i_reward_corr = [x["eval_i_reward_correlation"] for x in history if "eval_i_reward_correlation" in x]
    logp_corr = [x["eval_logp_correlation"] for x in history if "eval_logp_correlation" in x]
    
    if eval_epochs:
        ax2.plot(eval_epochs, i_reward_corr, label="Implicit Reward Corr", color="forestgreen", marker="s")
        ax2.plot(eval_epochs, logp_corr, label="Policy Log-P Corr", color="purple", marker="^")
        ax2.set_xlabel("Epoch")
        ax2.set_ylabel("Spearman Correlation")
        ax2.set_title("Evaluation Correlation Metrics")
        ax2.grid(True)
        ax2.legend()
        
    plt.tight_layout()
    plt.show()

plot_grpo_history(grpo_trainer)

torch.cuda.empty_cache()

## 12. Generate 1000 Sequences from the Fine-Tuned RL Model

To avoid Out-Of-Memory (OOM) errors on the GPU, we generate sequences in batches (e.g. batch size of 50) and output them to a clean CSV file.

In [ ]:
def generate_large_batch(total_sequences=1000, batch_size=50, max_length=100):
    generated_sequences = []
    
    # Standard starting template (e.g. "M" or other prompt)
    prompt = "M"
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    
    num_batches = (total_sequences + batch_size - 1) // batch_size
    print(f"Generating {total_sequences} sequences in {num_batches} batches...")
    
    for i in range(num_batches):
        current_batch_size = min(batch_size, total_sequences - len(generated_sequences))
        with torch.no_grad():
            gen_ids = model.generate(
                **inputs,
                max_new_tokens=max_length,
                do_sample=True,
                num_return_sequences=current_batch_size,
                top_p=0.95,
                temperature=1.0,
                pad_token_id=tokenizer.pad_token_id,
            )
        decoded = tokenizer.batch_decode(gen_ids, skip_special_tokens=True)
        # Clean up spaces to format as standard biological sequences
        cleaned = [seq.replace(" ", "") for seq in decoded]
        generated_sequences.extend(cleaned)
        print(f"Generated {len(generated_sequences)}/{total_sequences} sequences...")
        
    return generated_sequences

# Generate 1000 sequences
sequences = generate_large_batch(total_sequences=1000, batch_size=50)

# Save sequences to CSV
output_csv = "generated_rl_sequences.csv"
gen_df = pd.DataFrame({"sequence": sequences})
gen_df.to_csv(output_csv, index=False)
print(f"Saved 1000 generated sequences to {output_csv}")